In [1]:
# Imports and config
import os, json, numpy as np
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics.pairwise import rbf_kernel

# Ensure `src` package is importable when running from the notebooks/ folder
import sys, pathlib
try:
    p = pathlib.Path.cwd()
except Exception:
    p = pathlib.Path('/workspaces/Quantum-machine-learning-/qml-benchmark/notebooks')
candidate = p.parent
if (candidate / 'src').exists():
    sys.path.insert(0, str(candidate))
else:
    candidate2 = p.parent.parent / 'qml-benchmark'
    if (candidate2 / 'src').exists():
        sys.path.insert(0, str(candidate2))

from src.config import SEED, RESULTS_DIR
from src.projected_features import projected_phi

np.random.seed(SEED)
OUT_DIR = os.path.join(RESULTS_DIR, "projected_kernel_cv")
os.makedirs(OUT_DIR, exist_ok=True)


In [2]:
# Cell 2: Data
X = np.random.randn(200, 2)
y = ((X[:, 0] * X[:, 1]) > 0).astype(int)


In [3]:
# Cell 3: Grid and CV
grid = {"reps": [1, 2], "shots": [1000], "C": [0.1, 1.0, 10.0], "gamma": ["scale", 0.5, 1.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
param_grid = list(ParameterGrid(grid))
# --- IGNORE ---

In [4]:
# CV loop with φ(x), outer RBF, and optional φ-space Gram export
all_results = []
best_acc, best_cfg = -np.inf, None

for cfg in ParameterGrid(grid):
    accs, aucs = [], []
    for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        Phi_tr = projected_phi(X_tr, reps=cfg["reps"], shots=cfg["shots"])
        Phi_te = projected_phi(X_te, reps=cfg["reps"], shots=cfg["shots"])

        # Save features
        np.save(os.path.join(OUT_DIR, f"Phi_tr_f{fold}_r{cfg['reps']}.npy"), Phi_tr)
        np.save(os.path.join(OUT_DIR, f"Phi_te_f{fold}_r{cfg['reps']}.npy"), Phi_te)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"], random_state=SEED))
        ])
        pipe.fit(Phi_tr, y_tr)
        y_pred = pipe.predict(Phi_te)

        accs.append(accuracy_score(y_te, y_pred))
        try:
            aucs.append(roc_auc_score(y_te, y_pred))
        except Exception:
            aucs.append(np.nan)

        # Optional: φ-space Gram for precomputed-parity checks
        gamma_val = None if cfg["gamma"] == "scale" else cfg["gamma"]
        Kphi_tr = rbf_kernel(Phi_tr, Phi_tr, gamma=gamma_val)
        Kphi_te = rbf_kernel(Phi_te, Phi_tr, gamma=gamma_val)
        np.save(os.path.join(OUT_DIR, f"Kphi_tr_f{fold}_r{cfg['reps']}_g{cfg['gamma']}.npy"), Kphi_tr)
        np.save(os.path.join(OUT_DIR, f"Kphi_te_f{fold}_r{cfg['reps']}_g{cfg['gamma']}.npy"), Kphi_te)

    mean_acc = float(np.nanmean(accs))
    mean_auc = float(np.nanmean(aucs))
    all_results.append({"cfg": cfg, "acc": {"mean": mean_acc}, "auc": {"mean": mean_auc}})
    if mean_acc > best_acc:
        best_acc, best_cfg = mean_acc, cfg

with open(os.path.join(OUT_DIR, "cv_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)
with open(os.path.join(OUT_DIR, "best_cfg.json"), "w") as f:
    json.dump({"best_acc": best_acc, "best_cfg": best_cfg}, f, indent=2)

print("Best:", best_cfg, "Acc:", best_acc)


Best: {'C': 0.1, 'gamma': 'scale', 'reps': 2, 'shots': 1000} Acc: 0.52
